# Feature Engineering — простыми словами

Учебный ноутбук: как **создавать новые признаки** из уже имеющихся, чтобы модель лучше понимала данные.

Разбираем:
1. **Арифметические** признаки (отношения, разности…);
2. **PolynomialFeatures**;
3. **Логарифм** (`log1p`);
4. Признаки из **даты/времени**;
5. **Биннинг**;
6. **GroupBy-агрегаты** (и отличие от Target Encoding);
7. **Pivot / разворот** таблицы и **разбор** составного поля;
8. Коротко про **Featuretools** (Auto FE).

**Библиотеки:** `numpy`, `pandas`, `scikit-learn`, `matplotlib`.

Запускайте ячейки **сверху вниз**.

---

## План

1. [Словарь и общая идея](#dict)
2. [Арифметические признаки](#arith)
3. [Полиномиальные признаки](#poly)
4. [Логарифмирование](#log)
5. [Признаки из даты и времени](#date)
6. [Биннинг](#bin)
7. [GroupBy-признаки](#groupby)
8. [Pivot и разбор строки (как на слайде)](#reshape)
9. [Auto-ML: Featuretools](#auto)
10. [Шпаргалка](#итог)

> **Feature Engineering** = сознательное создание / преобразование признаков.  
> Это не «магия AutoML», а в первую очередь **смысл задачи** + проверка через **CV**.


<a id="dict"></a>
## 1. Словарь

| Термин | Простыми словами |
|--------|------------------|
| **Признак (feature)** | Столбец-вход для модели |
| **Feature Engineering** | Придумать / посчитать **новые** столбцы |
| **Гипотеза** | «Этот новый столбец *должен* помочь, потому что…» |
| **Pipeline** | Одинаковые преобразования на train / valid / test / проде |
| **Утечка (leakage)** | В train затекла информация из test (часто через groupby/mean «на всём») |
| **Target Encoding** | Среднее **y** по категории (опасно без защиты) |
| **GroupBy-признак** | Агрегат по **X** (mean income by city) — **без** y |

### Как проверять новый признак

```text
Понял данные → гипотеза → создал признак
        → Cross-Validation → оставил, только если качество выросло
```

**Антипаттерн:** «переберём все комбинации столбцов» → сотни мусорных признаков и переобучение.


<a id="arith"></a>
## 2. Арифметические признаки

Из нескольких столбцов — новый через **математику**.

### Зачем (пример кредита)

| Доход | Долг |
|------:|-----:|
| 50 000 | 20 000 |
| 500 000 | 20 000 |

Долг **одинаковый**, риск **разный**.  
Гораздо информативнее:

$$
\text{debt\_to\_income} = \frac{\text{debt}}{\text{income}}
$$

| Доход | Долг | Debt/Income |
|------:|-----:|------------:|
| 50 000 | 20 000 | **0.40** |
| 500 000 | 20 000 | **0.04** |

### Частые операции

| Операция | Пример | Как часто |
|----------|--------|-----------|
| **Деление** | debt/income, price/m², BMI | **самое частое** |
| **Вычитание** | revenue − costs, Δ температуры | очень часто |
| **Умножение** | price × qty, взаимодействия | часто |
| **Сложение** | еда + транспорт | реже |

### Главный секрет

Создавать **не потому что можно**, а когда есть **логический смысл**  
(финансы → DTI; недвижимость → цена/м²; медицина → BMI).

Источники идей: **предметная область** (главное) + EDA (графики, корреляции).

### Pipeline?

**Да**, если признак входит в обучение: одинаково на train/valid/test/проде  
(`FunctionTransformer`, свой Transformer, или заранее в маленьком проекте).

### Плюсы / минусы

**+** просто, интерпретируемо, дёшево, часто сильный буст.  
**−** нужен домен; легко наплодить мусор; корреляции между признаками.

### Где помогает

Практически **любые** модели (линейные, SVM, kNN, деревья, бустинги) — если признак несёт **новую** информацию.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer

# --- debt / income ---
df = pd.DataFrame({
    "income": [50_000, 500_000, 80_000, 120_000],
    "debt":   [20_000,  20_000, 40_000,  10_000],
    "default": [1, 0, 1, 0],  # игрушечный target
})
df["debt_to_income"] = df["debt"] / df["income"]
print(df.to_string(index=False))
print()

# Ручная проверка «помог ли признак» (очень маленький toy — только идея!)
X1 = df[["income", "debt"]].values
X2 = df[["income", "debt", "debt_to_income"]].values
y = df["default"].values

# На 4 объектах CV условна; в реале — больше данных
print("Идея проверки: сравнить CV с признаком и без (на реальных данных).")
print("Здесь dti:", df["debt_to_income"].round(3).tolist())

# FunctionTransformer в Pipeline (деление безопаснее с eps)
def add_dti(X):
    X = np.asarray(X, dtype=float)
    income, debt = X[:, 0], X[:, 1]
    dti = debt / np.clip(income, 1e-9, None)
    return np.column_stack([X, dti])

pipe = Pipeline([
    ("arith", FunctionTransformer(add_dti)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
# fit на toy просто чтобы показать API
pipe.fit(df[["income", "debt"]].values, y)
print("Pipeline с DTI обучился. predict_proba[0] =",
      np.round(pipe.predict_proba(df[["income", "debt"]].values)[0], 3))


<a id="poly"></a>
## 3. Полиномиальные признаки (Polynomial Features)

**Автоматически** добавляют **степени** и **произведения**, чтобы **линейная** модель могла описать **нелинейность**.

### Идея

Истина: $y = x^2$ (парабола).  
Линейная модель: $y = b_0 + b_1 x$ — только прямая.  
Добавили $x^2$: $y = b_0 + b_1 x + b_2 x^2$ — уже парабола.

### Пример degree=2, признаки A, B

$$
A,\; B,\; A^2,\; B^2,\; A\cdot B
$$

(при `include_bias=False` без столбца единиц).

### Код

```python
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
```

| Параметр | Смысл |
|----------|--------|
| `degree` | степень (на практике **2**, реже **3**, выше почти никогда) |
| `include_bias=True` | столбец единиц; у `LinearRegression` intercept уже есть → почти всегда **`False`** |

### Pipeline (типично)

```python
Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression()),
])
```

> **Порядок:** часто **сначала scale, потом poly** (или poly на сырых — зависит от задачи; scale до poly стабилизирует произведения).  
> Главное — **один и тот же** порядок в train и test через Pipeline.

### Когда да / нет

| Модели | Polynomial Features |
|--------|---------------------|
| Linear / Ridge / Lasso / LogReg / SVM | часто **да** |
| Tree / RF / XGB / LGBM / CatBoost | обычно **нет** — деревья сами ловят нелинейности |

### Комбинаторный взрыв (`include_bias=False`, degree=2)

Число признаков ≈ $n(n+3)/2$:

| Исходных $n$ | После degree=2 |
|-------------:|---------------:|
| 10 | 65 |
| 20 | 230 |
| 100 | 5150 |

→ риск **переобучения** и долгий fit. Поэтому не ставят degree=10 «на всякий случай».

### FE вручную vs PolynomialFeatures

| | Ручной FE | PolynomialFeatures |
|--|-----------|-------------------|
| Основа | **домен** | только математика |
| Объём | мало сильных | много, часть мусор |
| Кому помогает | почти всем моделям | в основном **линейным** |

Не «вместо», а **разные задачи**.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

np.random.seed(0)
x = np.linspace(-2, 2, 60)
y = x ** 2 + np.random.normal(0, 0.15, size=len(x))
X = x.reshape(-1, 1)

# Линейная без poly vs с poly
pipe_lin = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
pipe_poly = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression()),
])

cv_lin = -cross_val_score(pipe_lin, X, y, cv=5, scoring="neg_mean_squared_error").mean()
cv_poly = -cross_val_score(pipe_poly, X, y, cv=5, scoring="neg_mean_squared_error").mean()
print(f"CV MSE линейная:     {cv_lin:.4f}")
print(f"CV MSE + poly deg=2: {cv_poly:.4f}")

poly = PolynomialFeatures(degree=2, include_bias=False)
print("Имена признаков:", poly.fit(X).get_feature_names_out(["x"]))

# Рост числа признаков
def n_poly(n, degree=2):
    return PolynomialFeatures(degree=degree, include_bias=False).fit(
        np.zeros((2, n))
    ).n_output_features_

print("\nСколько столбцов при degree=2:")
for n in [2, 10, 20, 100]:
    print(f"  n={n:3d} → {n_poly(n)}")


<a id="log"></a>
## 4. Логарифмирование признаков

```python
df["log_income"] = np.log(df["income"])     # только x > 0
df["log_income"] = np.log1p(df["income"])   # log(1+x), удобно при нулях — **предпочтительнее**
```

### Зачем

Доходы: 30k, 50k, …, **5 000 000** — «хвост» вправо давит на линейные модели и расстояния.  
Логарифм **сжимает** большие значения: рост $x$ в 10 раз → $\log_{10} x$ +1 (для натурального log — сдвиг на $\ln 10$).

### Когда

Сильная **правосторонняя** асимметрия: зарплаты, цены, просмотры, подписчики.  
Смотрите **гистограмму** до/после.

### Pipeline

```python
FunctionTransformer(np.log1p)
```

Часто **потом** `StandardScaler`: log меняет **форму**, scaler — **масштаб**. Это **разное**.

### Когда не надо

Уже «колокол»; есть **отрицательные** (обычный log нельзя — нужен другой transform); выгоды нет по CV.

### Модели

Сильнее выигрывают **линейные / SVM / kNN**.  
Деревьям форма распределения менее критична, но log-признак иногда всё равно полезен.


In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
# логнормаль ≈ «доходы»
income = rng.lognormal(mean=10.5, sigma=0.8, size=2000)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(income, bins=40, color="steelblue", edgecolor="white")
ax[0].set_title("income (сырой) — длинный правый хвост")
ax[1].hist(np.log1p(income), bins=40, color="seagreen", edgecolor="white")
ax[1].set_title("log1p(income) — ближе к симметрии")
plt.tight_layout(); plt.show()

print("max/median raw:", round(income.max()/np.median(income), 1))
print("max/median log1p:", round(np.log1p(income).max()/np.median(np.log1p(income)), 2))


<a id="date"></a>
## 5. Признаки из даты и времени

Один столбец `2026-08-07 14:35` → год, месяц, день, **день недели**, час, квартал, выходной, «дней с регистрации»…

### Почему помогает

Модель плохо ест «голое» число `20260807`.  
А «суббота» / «час 21» часто связаны с покупками, нагрузкой, мошенничеством.

### Типичный код

```python
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.weekday   # 0=пн … 6=вс
df["hour"] = df["date"].dt.hour
df["is_weekend"] = (df["weekday"] >= 5).astype(int)
df["quarter"] = df["date"].dt.quarter
df["days_since_reg"] = (today - df["registration_date"]).dt.days
```

### Число 0…6 для weekday — проблема для линейных моделей

Для линейной регрессии «воскресенье=6» ≠ «в 6 раз больше понедельника».  
Обычно:

| Модель | weekday / month / hour |
|--------|-------------------------|
| Linear / LogReg / SVM / kNN | **One-Hot** (или аналог) |
| Tree / RF / Boosting | часто **оставить числом** |
| CatBoost | категории «из коробки» |

### Цикличность (час 23 и 00)

Числа 23 и 0 «далеко», хотя по времени рядом.  
**Cyclical encoding:**

$$
\sin\bigl(2\pi \cdot h / 24\bigr),\quad \cos\bigl(2\pi \cdot h / 24\bigr)
$$

Нужно в рядах/нагрузке/погоде; в обычном tabular чаще хватает One-Hot / деревьев.


In [ ]:
# Дата → признаки
df_d = pd.DataFrame({
    "purchase_date": [
        "2026-08-07 14:35",
        "2026-08-08 21:10",
        "2026-08-09 09:00",
        "2026-08-10 23:50",
    ],
    "amount": [100, 250, 80, 400],
})
df_d["purchase_date"] = pd.to_datetime(df_d["purchase_date"])
df_d["weekday"] = df_d["purchase_date"].dt.weekday
df_d["hour"] = df_d["purchase_date"].dt.hour
df_d["is_weekend"] = (df_d["weekday"] >= 5).astype(int)

# cyclical hour
df_d["hour_sin"] = np.sin(2 * np.pi * df_d["hour"] / 24)
df_d["hour_cos"] = np.cos(2 * np.pi * df_d["hour"] / 24)
print(df_d[["purchase_date", "weekday", "hour", "is_weekend", "hour_sin", "hour_cos"]].to_string(index=False))

# 23:00 и 00:00 близки в sin/cos-пространстве
h23 = np.array([np.sin(2*np.pi*23/24), np.cos(2*np.pi*23/24)])
h00 = np.array([np.sin(2*np.pi*0/24), np.cos(2*np.pi*0/24)])
h12 = np.array([np.sin(2*np.pi*12/24), np.cos(2*np.pi*12/24)])
print("\nЕвклидово расстояние hour-features:")
print("  23 vs 00:", round(np.linalg.norm(h23 - h00), 3), "(близко)")
print("  23 vs 12:", round(np.linalg.norm(h23 - h12), 3), "(далеко)")


<a id="bin"></a>
## 6. Биннинг (дискретизация)

Непрерывное число → **корзины** (группы).

| Возраст | Группа |
|--------:|--------|
| 18 | 18–25 |
| 27 | 26–35 |
| 63 | 56+ |

### Зачем

Нелинейная «полка» покупок по возрасту линейной модели трудна;  
**группам** можно дать разные коэффициенты (особенно в скоринге).

### Способы (`KBinsDiscretizer`)

| strategy | Идея |
|----------|------|
| `"uniform"` | равная **ширина** интервалов |
| `"quantile"` | равное **число объектов** — часто лучший default |
| `"kmeans"` | границы через KMeans — реже |

```python
KBinsDiscretizer(n_bins=5, strategy="quantile", encode="ordinal")  # 0..4
# encode="onehot" / "onehot-dense" — сразу one-hot
```

### Pipeline — да

### Когда

Скоринг, страхование, маркетинг, медицина + **линейные/логистические** модели.

### Когда нет

RF / XGB / CatBoost — деревья **сами** ищут пороги; биннинг часто лишний и **теряет** точность (29 vs 30 в разных бинах).

### Ручной vs авто

Бизнес-правила «до 21 / 21–60 / 60+» могут быть **важнее**, чем математически «оптимальные» квантили — как с ручным FE vs PolynomialFeatures.


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

age = np.array([18, 21, 27, 35, 48, 63, 22, 40, 55, 70], dtype=float).reshape(-1, 1)

for strategy in ["uniform", "quantile"]:
    kb = KBinsDiscretizer(n_bins=4, strategy=strategy, encode="ordinal", subsample=None)
    bins = kb.fit_transform(age).ravel()
    edges = kb.bin_edges_[0]
    print(f"{strategy}: edges={np.round(edges, 1)}")
    print(f"  ages={age.ravel()} → bins={bins.astype(int)}")
    print()


<a id="groupby"></a>
## 7. GroupBy-признаки (агрегаты по группе)

К каждой строке добавить статистику **её группы** (город, магазин, категория…).

| Клиент | Город | Доход | **Средний доход города** |
|--------|-------|------:|-------------------------:|
| A | Москва | 100 | 100 |
| B | Москва | 120 | 100 |
| C | Москва | 80 | 100 |
| D | СПб | 60 | 65 |
| E | СПб | 70 | 65 |

Модель видит не только человека, но и **контекст** («богатый регион»).

### Частые агрегаты

`mean`, `median`, `max`, `min`, `std`, `count`  

Очень сильный ход — **отклонение от среднего группы**:

```python
df["city_mean_income"] = df.groupby("city")["income"].transform("mean")
df["income_diff"] = df["income"] - df["city_mean_income"]
```

### Почему `transform`, не `mean()`?

`groupby(...).mean()` → **мало строк** (по одной на город).  
`transform("mean")` → **столько же строк**, сколько было (среднее «размазано» на членов группы).

### Утечка данных

**Нельзя** считать `groupby` mean по **всему** датасету до split, если в «всём» есть test:  
статистика test **затечёт** в train.

**Правильно:**

1. Статистики **только по train**;  
2. Применить map к train и test;  
3. В K-Fold — статистики **только внутри train_fold** (или аккуратный Transformer).

### GroupBy-признаки ≠ Target Encoding

| | GroupBy по **X** | Target Encoding |
|--|------------------|-----------------|
| Что усредняем | признак (доход, возраст…) | **target y** |
| Подглядывание y | **нет** | **да** → нужна защита (CV target enc…) |
| Пример | mean income by city | mean default rate by city |

Внешне похоже, по риску leakage — **разное**.

### Где помогает

Почти все модели, если есть осмысленные **группы**.


In [ ]:
# GroupBy + transform
clients = pd.DataFrame({
    "client": list("ABCDE"),
    "city": ["Москва", "Москва", "Москва", "СПб", "СПб"],
    "income": [100, 120, 80, 60, 70],
})
clients["city_mean_income"] = clients.groupby("city")["income"].transform("mean")
clients["income_diff"] = clients["income"] - clients["city_mean_income"]
print(clients.to_string(index=False))
print()

# Честный train/test: статистики только с train
train = clients.iloc[:3].copy()   # только Москва в этом toy
test = clients.iloc[3:].copy()    # СПб

city_means = train.groupby("city")["income"].mean()  # fit на train
print("city_means с train:", city_means.to_dict())

# map; для новых городов — fallback (глобальное mean train)
global_mean = train["income"].mean()
for part_name, part in [("train", train), ("test", test)]:
    part = part.copy()
    part["city_mean_income_safe"] = part["city"].map(city_means).fillna(global_mean)
    print(f"{part_name}:")
    print(part[["client", "city", "income", "city_mean_income_safe"]].to_string(index=False))
    print()
print("Замечание: в test города из train map'ятся; новый город → global_mean train.")


<a id="reshape"></a>
## 8. Pivot и разбор составного поля (как на учебном слайде)

### 8.1. Группировка / pivot: длинная таблица → широкая

**Задача со слайда:** сгруппировать по **пользователю**, города сделать **столбцами**,  
на пересечении — сумма **дней посещений**.

**Было (long):**

| User | City | Visit days |
|-----:|------|----------:|
| 1 | Roma | 1 |
| 2 | Madrid | 2 |
| 1 | Madrid | 1 |
| 3 | Istanbul | 1 |
| 2 | Istanbul | 4 |
| 1 | Istanbul | 3 |
| 1 | Roma | 3 |

**Стало (wide):**

| User | Istanbul | Madrid | Roma |
|-----:|---------:|-------:|-----:|
| 1 | 3 | 1 | **4** |  ← Roma: 1+3 |
| 2 | 4 | 2 | 0 |
| 3 | 1 | 0 | 0 |

Это классический **pivot / pivot_table** (иногда «группировка с разворотом»).


In [ ]:
# Данные со слайда
visits = pd.DataFrame({
    "User": [1, 2, 1, 3, 2, 1, 1],
    "City": ["Roma", "Madrid", "Madrid", "Istanbul", "Istanbul", "Istanbul", "Roma"],
    "Visit days": [1, 2, 1, 1, 4, 3, 3],
})
print("Long format:")
print(visits.to_string(index=False))

wide = visits.pivot_table(
    index="User",
    columns="City",
    values="Visit days",
    aggfunc="sum",
    fill_value=0,
).reset_index()
# порядок столбцов как на слайде
wide = wide[["User", "Istanbul", "Madrid", "Roma"]]
print("\nWide (pivot):")
print(wide.to_string(index=False))

# Эквивалент через groupby + unstack
wide2 = (
    visits.groupby(["User", "City"])["Visit days"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)
print("\nПроверка User=1 Roma (1+3):", int(wide.loc[wide["User"]==1, "Roma"].iloc[0]))


### 8.2. Разделение одного поля на несколько

**Было:** одно поле `Policy` = тип + уровень в одной строке.

| Policy |
|--------|
| Corporate L3 |
| Personal L3 |
| Personal L3 |
| Corporate L2 |
| Personal L3 |

**Стало:**

| Type | Level |
|------|-------|
| Corporate | L3 |
| Personal | L3 |
| Personal | L3 |
| Corporate | L2 |
| Personal | L3 |

Типичные приёмы: `str.split`, регулярные выражения, `str.extract`.


In [ ]:
# Разбор "Corporate L3" → Type + Level
policy = pd.DataFrame({
    "Policy": [
        "Corporate L3",
        "Personal L3",
        "Personal L3",
        "Corporate L2",
        "Personal L3",
    ]
})

# Вариант 1: split по пробелу (если формат стабильный)
parts = policy["Policy"].str.split(n=1, expand=True)
policy["Type"] = parts[0]
policy["Level"] = parts[1]

# Вариант 2: regex (гибче)
extracted = policy["Policy"].str.extract(r"^(?P<Type>\w+)\s+(?P<Level>L\d+)$")
print(policy[["Policy", "Type", "Level"]].to_string(index=False))
print("\nregex совпал:", extracted.equals(policy[["Type", "Level"]]))


<a id="auto"></a>
## 9. Автоматическая генерация: Featuretools

**Featuretools** — open-source библиотека **автоматического** Feature Engineering  
(много сущностей, связи client→transactions→…, «глубина» агрегатов).

Идея со слайда: работает рядом с **pandas** и **scikit-learn**, ускоряет создание признаков,  
которые вручную заняли бы много времени.

| | Ручной FE | Featuretools / AutoFE |
|--|-----------|------------------------|
| Контроль | полный | меньше |
| Смысл | вы задаёте | много кандидатов, нужен отбор |
| Риск | мало мусора, если думать | легко получить **тысячи** столбцов |

На практике: AutoFE как **помощник**, не замена пониманию данных.  
Установка при необходимости: `pip install featuretools` (в этом ноутбуке не обязательна).

> После автогенерации всё равно: **CV**, отбор признаков, проверка leakage.


<a id="итог"></a>
## 10. Самое главное + шпаргалка

| Техника | Суть | Типичный выигрыш |
|---------|------|------------------|
| **Арифметика** | + − × ÷ с **смыслом** | все модели |
| **Polynomial** | степени/произведения | линейные |
| **log1p** | сжать правый хвост | линейные, kNN, SVM |
| **Дата** | year/month/weekday/hour… | почти все |
| **Биннинг** | число → группы | logreg/скоринг |
| **GroupBy** | контекст группы по **X** | почти все |
| **Pivot / split** | форма таблицы / разбор строк | подготовка данных |
| **Featuretools** | авто-кандидаты | при аккуратном отборе |

### Железные правила

1. Сначала **гипотеза**, потом признак, потом **CV**.  
2. Одинаковые преобразования на train и test (**Pipeline** / общий map).  
3. Groupby-статистики — **только с train** (иначе leakage).  
4. Target Encoding ≠ groupby по X.  
5. Не плодить degree=10 и «все пары столбцов» без нужды.  
6. Для деревьев poly/binning часто **лишние**; отношения и groupby — нет.

### Мини-глоссарий

| | |
|--|--|
| **DTI** | debt/income |
| **include_bias** | столбец 1 в PolynomialFeatures |
| **log1p** | $\log(1+x)$ |
| **transform (groupby)** | агрегат той же длины, что таблица |
| **pivot_table** | long → wide |


### Мини-практика

1. Добавьте `price_per_m2 = price / area` и сравните CV линейной модели с/без.  
2. На $y \approx x^2$ сравните LinearRegression с `PolynomialFeatures(degree=2)`.  
3. На exp-доходах сравните гистограммы raw vs `log1p`.  
4. Сделайте `pivot_table` по данным «User / City / Visit days» со слайда.  
5. Разберите `Policy` на Type и Level через `str.split`.  
6. Посчитайте `city_mean` **только на train** и примените к test.


In [ ]:
# ===== Мини-практика: эталонные сниппеты =====
print("1) price/m2")
re = pd.DataFrame({"price": [6e6, 9e6, 12e6], "area": [40, 50, 80]})
re["price_per_m2"] = re["price"] / re["area"]
print(re)

print("\n4) pivot уже выше; быстрый повтор:")
print(wide.to_string(index=False))

print("\n5) split Policy — OK")
print(policy[["Type", "Level"]].head(2).to_string(index=False))

print("\nГлавное: гипотеза → признак → CV → оставить или выбросить.")


## Что делать дальше

1. Свяжите с **валидацией**: любой FE, считающий статистики, — только на train_fold.  
2. Свяжите с **пропусками**: сначала честная импутация, потом FE (или наоборот осознанно).  
3. Свяжите с **метриками**: улучшение смотрите по valid/CV, не по train RMSE «в ноль».

### Главная мысль

> Хороший признак — это **сжатая экспертиза** о мире  
> (отношение, контекст группы, сезонность),  
> а не случайная формула.  
> PolynomialFeatures и AutoFE — инструменты, а не замена мышлению.

Удачи!
